# 多模态大模型全景：从视觉理解到生成与 Omni

> **本章定位**：建立多模态理解、Omni 交互、文生图、文生视频与世界模型的任务地图，并以 Vision-Language Model（视觉语言模型，VLM）为最小原理实现主线。

> **章节边界**：本章属于跨方向专题：多模态理解，以 `31` 的语言解码器和 `E40` 的视觉编码器为基础，聚焦感知型 VLM 的数据流与接口契约；生成视觉的数学与实现由 `E50_cv_diffusion.ipynb` 展开。

**本章总览**：内容依次覆盖多模态任务分类、视觉 Token 连接器、多模态序列与标签掩码、分阶段训练、标准库迁移以及任务级生产验收。

```mermaid
flowchart TD
    X["Text / Image / Video / Audio / Action"] --> R{"任务输出是什么？"}
    R -->|"Text / JSON / Tool Call"| U["多模态理解 VLM"]
    R -->|"Text + Streaming Speech"| O["Omni 交互模型"]
    R -->|"Image / Edit"| I["图像生成模型"]
    R -->|"Video / Audio / Future State"| V["视频与世界模型"]
    U --> M["本章原理实现主线"]
    I --> E["E50：Diffusion / Flow"]
    V --> E
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 跨方向专题：多模态理解 |
| 本章定位 | 建立多模态任务与模型图谱，并以视觉 Encoder 接入 Decoder-only LLM 作为实现主线。 |
| 先修知识 | 掌握 `31` 的语言解码器和 `E40` 的视觉编码器。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | 最小组件可在 CPU 运行；真实 VLM 建议使用 GPU。 |
| 输入 | 模型卡与许可证，以及图像、文本 Token、视觉 Token 和 Label Mask。 |
| 交付物 | 多模态模型图谱、最小视觉连接器、标准 Processor/模型接口和任务级验收矩阵。 |

### 1.1．学习目标

完成本章后，读者能够按输入与输出模态区分理解、Omni 和生成任务，解释视觉 Token 接入语言模型的数据流，并构造 Attention Mask 与 Label Mask。原理实现覆盖视觉编码器、Projector、Resampler 和多模态序列拼接；生成模型的去噪过程由 `E50_cv_diffusion.ipynb` 承接。


## 2．直觉与输入输出契约

“多模态”只说明涉及多个模态，不说明输出能力。模型选型首先需要明确 `输入模态 → 输出模态` 的接口契约：

| 类型 | 典型输入 → 输出 | 常见核心 | 不能据此推断 |
|---|---|---|---|
| 感知型 VLM | 图像/视频 + 文本 → 文本/JSON/Tool Call | 视觉编码器 + Connector/Fusion + LLM | 会生成图像或视频像素 |
| Omni 交互 | 文本/图像/视频/音频 → 文本和/或流式语音 | 多编码器或统一 Token + LLM + Speech Decoder | 能输出所有输入模态 |
| 图像生成/编辑 | 文本/参考图 → 图像 | VAE/Tokenizer + U-Net/DiT/自回归视觉 Token | 能可靠回答视觉事实问题 |
| 视频生成 | 文本/图像 → 视频，可能附音频 | 时空 VAE + Video DiT/Flow + Temporal Attention | 长时身份、物理与时序必然一致 |
| 世界模型 | 文本/视觉/动作 → 未来状态/视频/动作 | 感知、生成与动作条件联合建模 | 可直接用于安全关键控制 |

### 2.1．感知型 VLM 的三种主流融合边界

| 方案 | 数据流 | 优点 | 代价 |
|---|---|---|---|
| Projector Prefix | 视觉 Token 投影后作为文本前缀 | 结构简单、易复用 Decoder-only LLM | 高分辨率会消耗大量上下文 |
| Q-Former / Resampler | 固定数量 Query 从视觉 Token 提取信息 | 控制视觉 Token 数，适合多图 | 连接器本身需要训练 |
| Cross-Attention | 文本层显式读取视觉 Memory | 模态边界清晰、可按层注入 | 需要改造 LLM Block 和推理 Kernel |

本章采用 `Projector + Resampler + Prefix` 的原理实现，以显式呈现维度对齐、Token 压缩和 Label Mask 三项关键契约。

<!-- diagram:multimodal-fusion -->

![架构图：视觉 Patch Token 经投影和查询 Resampler 压缩后作为 LLM 前缀](assets/figures/E20_multimodal_llm/multimodal-fusion.svg)

[TikZ 源文件](assets/figures/E20_multimodal_llm/multimodal-fusion.tex)


### 2.2．理解与 Omni 模型图谱（2026-08-09 快照）

下表列出代表性模型入口，用于比较接口与架构差异，不构成能力排序。许可证、上下文与运行时支持应在发布前按当前文件及其哈希复核。

| 地区/家族 | 输入 → 输出与架构视角 | 开放状态 | 生产边界 |
|---|---|---|---|
| 中国｜**Qwen3-VL** | 图像/视频 + 文本 → 文本；Dense/MoE、Interleaved-MRoPE、DeepStack、时间戳对齐 | Apache-2.0 开放权重 | 不生成像素；长视频需帧采样、视觉 Token 预算与分段；GUI Agent 加权限和结果校验 |
| 中国｜**Qwen3-Omni** | 文本/图像/音频/视频 → 文本 + 实时语音；MoE Thinker–Talker | Apache-2.0 开放权重 | Omni 不等于图像/视频输出；流媒体需分块、背压、并发隔离与声音冒充治理 |
| 中国｜**Kimi-VL → Kimi K3** | 轻量 MoE VLM 到数据中心级原生视觉 Agent；MoonViT、MLA/KDA 与 MoE | Kimi-VL 为 MIT；K3 为自定义 Kimi K3 License | 不可把旧代许可套到 K3；旗舰总权重、长上下文和视觉 Token 都需多机预算 |
| 中国｜**InternVL3.5** | 图像/视频 → 文本；动态分辨率、视觉路由、ViT–MLP–LLM | 仓库 MIT，检查点还受底层 LLM 条款约束 | 组合模型必须递归核验所有组件许可证；具身/GUI 动作必须经受控执行器 |
| 中国｜**Janus-Pro** | 图像理解 + 文生图；解耦理解/生成视觉编码器，共享自回归 Transformer | 代码 MIT，权重为 DeepSeek Model License | 适合分析统一建模路线；其分辨率与画质目标不同于通用高分辨率生图引擎 |
| 美国｜**Llama 4** | 图像 + 文本 → 文本/代码；Early Fusion 原生多模态 MoE | Llama Community License + AUP | 不是 Apache/MIT；地域、规模、分发和衍生条款需单独审查 |
| 美国｜**Gemma 4** | 文本/图像/视频，部分规格含音频 → 文本；Dense/MoE、局部/全局混合注意力 | Apache-2.0 开放权重 | 仅文本输出；不同规格的模态、上下文和编码器设计不同 |
| 美国｜**Phi-4 Multimodal** | 文本/图像/音频 → 文本；Phi-4-mini + 视觉/语音编码器与 Mixture-of-LoRAs | MIT 开放权重 | 紧凑不代表全语言/全任务；按视觉、语音和知识边界分别评测 |

趋势：Qwen3.6、Kimi K3、Gemma 4 等把视觉能力并入通用主干；这不改变接口审计原则——只有模型卡声明并通过验收的输入/输出模态才算产品能力。

### 2.3．生成模型的架构坐标

| 路线 | 数据流 | 代表家族 | 核心成本/风险 |
|---|---|---|---|
| 自回归视觉 Token | 文本/图像 Token → 逐 Token 图像 | Janus-Pro 等统一模型 | 离散 Tokenizer 质量、序列长度、采样延迟 |
| Latent Diffusion / Flow | 文本条件 + 噪声 Latent → U-Net/DiT 迭代 → 图像 | Qwen-Image、FLUX、Stable Diffusion | 步数、分辨率、文字/布局、VAE 与许可证 |
| Video DiT / Flow | 文本/首帧 + 时空 Latent → 视频 | Wan、HunyuanVideo、CogVideoX、Mochi | 帧数×分辨率带来的时空 Token、运动/身份/物理一致性 |
| 世界/音视频联合模型 | 文本/视觉/动作 → 视频、声音、动作或未来状态 | NVIDIA Cosmos、LTX | 声画同步、动力学错误、闭环安全与数据治理 |

生成模型的具体检查点、许可证和验收表进入 `E50_cv_diffusion.ipynb`。Sora、Veo、GPT 图像、Runway 等闭源/API 生成产品只能放在市场坐标，不能列入开放权重清单。


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# seed=42 只固定初始化与输入；正式质量结论采用预注册多 Seed 分布，且不承诺跨后端逐 bit 一致。
SEED = 42
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

# 32×32 图像按 8×8 Patch 切成 16 个视觉 Token；分辨率升高或 Patch 变小会按面积增加 Prefill 与显存。
IMAGE_SIZE = 32
PATCH_SIZE = 8
# Projector 将 vision dim=32 对齐到 LLM dim=48；真实检查点必须服从 Connector 配置。
VISION_DIM = 32
LLM_DIM = 48
# 48 可被 4 个 Head 整除；4 个 Query 将 16 个视觉 Token 压缩为四分之一，Query 数变化需重新验证质量与 TTFT。
NUM_HEADS = 4
NUM_QUERIES = 4
# -100 沿用 CrossEntropy 忽略协议，使视觉与 Prompt 可被读取但不进入语言损失。
IGNORE_INDEX = -100

DEVICE


<!-- theory-math-contract:v1 -->
### 2.4．核心机制的语言与数学表达

感知型视觉语言模型先把图像切成视觉 Token，再通过 Projector 对齐到语言模型隐藏维度并与文本 Token 拼接：

$$
N_v=\frac{H}{P_h}\frac{W}{P_w},\qquad
Z_v=f_{\mathrm{proj}}\!\left(f_{\mathrm{vision}}(I)\right)\in\mathbb{R}^{B\times N_v\times D_{\mathrm{llm}}}
$$

其中，$I\in\mathbb{R}^{B\times C\times H\times W}$，Patch 大小为 $P_h\times P_w$，$N_v$ 是未压缩视觉 Token 数。拼接后的序列长度为 $L=N_v+L_t+L_{\mathrm{special}}$；Label Mask 通常只让回答文本参与损失。`vision_encoder`、`projector/resampler`、`input_ids/inputs_embeds` 分别对应视觉编码、维度/长度对齐与语言模型输入。分辨率提高会以面积速度增加 Token，不等价于信息同比增加。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．视觉编码与 Patch Token 表示

卷积核和步幅都等于 Patch Size 时，一次卷积即可完成不重叠切块与线性投影。输出从 `[B, C, H, W]` 变为 `[B, Nv, Dv]`：

$$
N_v = (H / P) × (W / P)
$$


In [ ]:
class MyPatchVisionEncoder(nn.Module):
    """把 RGB 图像切分为不重叠 Patch Token，并加入可学习位置嵌入与归一化。"""
    def __init__(self, image_size: int, patch_size: int, hidden_size: int):
        """创建步幅等于 Patch 大小的卷积投影、位置参数和 LayerNorm。"""
        super().__init__()
        patches_per_side = image_size // patch_size
        self.num_patches = patches_per_side ** 2
        self.patch_embedding = nn.Conv2d(
            in_channels=3,
            out_channels=hidden_size,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.position_embedding = nn.Parameter(torch.zeros(1, self.num_patches, hidden_size))
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """将 [B,3,H,W] 图像编码为 [B,N_patch,D_vision] 视觉 Token。"""
        patch_grid = self.patch_embedding(images)
        patch_tokens = patch_grid.flatten(2).transpose(1, 2)
        return self.norm(patch_tokens + self.position_embedding)


# batch=2、RGB=3 覆盖图像张量四条轴；生产通道与尺寸由 Processor 契约确定。
# 固定形状：images.shape = [2, 3, 32, 32]。
images = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
vision_encoder = MyPatchVisionEncoder(IMAGE_SIZE, PATCH_SIZE, VISION_DIM).to(DEVICE)
vision_tokens = vision_encoder(images)
vision_tokens.shape


### 3.2．Projector 与 Resampler 的维度及长度变换

Projector 把 `Dv` 映射到 `Dl`；Resampler 使用固定数量的 Learned Query 读取所有视觉 Token，将视觉上下文长度固定为 `Nq`。这不是简单平均池化：不同 Query 可以学习关注不同区域或语义。


In [ ]:
class MyVisualResampler(nn.Module):
    """将可变数量视觉 Token 投影到语言空间，并用可学习 Query 压缩为固定长度视觉前缀。"""
    def __init__(self, vision_dim: int, llm_dim: int, num_heads: int, num_queries: int):
        """创建视觉投影器、可学习 Query、交叉注意力和输出归一化。"""
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
        self.queries = nn.Parameter(torch.randn(1, num_queries, llm_dim) / math.sqrt(llm_dim))
        self.cross_attention = nn.MultiheadAttention(llm_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(llm_dim)

    def forward(self, vision_tokens: torch.Tensor) -> torch.Tensor:
        """把 [B,N,D_vision] 视觉 Token 重采样为 [B,Q,D_llm] 固定长度前缀。"""
        memory = self.projector(vision_tokens)
        queries = self.queries.expand(memory.size(0), -1, -1)
        attended, _ = self.cross_attention(queries, memory, memory, need_weights=False)
        return self.norm(queries + attended)


resampler = MyVisualResampler(VISION_DIM, LLM_DIM, NUM_HEADS, NUM_QUERIES).to(DEVICE)
visual_prefix = resampler(vision_tokens)
visual_prefix.shape


### 3.3．多模态序列与 Label Mask

Prefix 融合后的序列是：

```text
[visual_1 ... visual_Nq] [bos] [prompt ...] [answer ...] [eos]
```

视觉位置没有词表 ID，不能作为语言建模标签；Prompt 是否监督取决于训练契约。指令微调通常只监督 Answer，因此视觉与 Prompt 位置都填 `-100`。Attention Mask 仍标记这些位置有效，因为后续文本必须能够读取视觉和 Prompt 上下文。


In [ ]:
@dataclass
class MyMultimodalOutput:
    """承载多模态因果语言模型的序列 logits 与可选训练损失。"""
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


class MyMultimodalCausalLM(nn.Module):
    """连接视觉编码器、重采样器与因果文本解码器，并对视觉前缀和屏蔽标签排除监督。"""
    # max_text_length=64 仅容纳本章短文本；真实上限、截断率与位置容量由模型和 Tokenizer 共同约束。
    def __init__(self, vocab_size: int, max_text_length: int = 64):
        """创建视觉前端、Token/位置嵌入、两层因果解码器及权重共享语言模型头。"""
        super().__init__()
        self.vision_encoder = MyPatchVisionEncoder(IMAGE_SIZE, PATCH_SIZE, VISION_DIM)
        self.resampler = MyVisualResampler(VISION_DIM, LLM_DIM, NUM_HEADS, NUM_QUERIES)
        self.token_embedding = nn.Embedding(vocab_size, LLM_DIM)
        self.position_embedding = nn.Embedding(NUM_QUERIES + max_text_length, LLM_DIM)
        # 两层 Decoder 控制实验成本；FFN 取 4×48=192，dropout=0.1 为训练基线，推理与等价检查应切换 eval。
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=LLM_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=4 * LLM_DIM,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=2)
        self.final_norm = nn.LayerNorm(LLM_DIM)
        self.lm_head = nn.Linear(LLM_DIM, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def forward(
        self,
        images: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> MyMultimodalOutput:
        """融合图像与文本序列并返回 [B,Q+T,V] logits；提供 labels 时计算移位交叉熵损失。"""
        visual_tokens = self.resampler(self.vision_encoder(images))
        text_tokens = self.token_embedding(input_ids)
        hidden_states = torch.cat([visual_tokens, text_tokens], dim=1)

        positions = torch.arange(hidden_states.size(1), device=hidden_states.device)
        hidden_states = hidden_states + self.position_embedding(positions)[None, :, :]

        visual_mask = torch.ones(
            attention_mask.size(0), NUM_QUERIES, dtype=attention_mask.dtype, device=attention_mask.device
        )
        multimodal_mask = torch.cat([visual_mask, attention_mask], dim=1)
        causal_mask = torch.triu(
            torch.ones(hidden_states.size(1), hidden_states.size(1), device=hidden_states.device, dtype=torch.bool),
            diagonal=1,
        )
        hidden_states = self.decoder(
            hidden_states,
            mask=causal_mask,
            src_key_padding_mask=~multimodal_mask.bool(),
        )
        logits = self.lm_head(self.final_norm(hidden_states))

        loss = None
        if labels is not None:
            visual_labels = torch.full(
                (labels.size(0), NUM_QUERIES), IGNORE_INDEX, dtype=labels.dtype, device=labels.device
            )
            multimodal_labels = torch.cat([visual_labels, labels], dim=1)
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                multimodal_labels[:, 1:].reshape(-1),
                ignore_index=IGNORE_INDEX,
            )
        return MyMultimodalOutput(logits=logits, loss=loss)


# vocab=128、文本长度=12 只用于验证 logits 形状；真实词表和上下文长度属于模型制品契约。
VOCAB_SIZE = 128
TEXT_LENGTH = 12
model = MyMultimodalCausalLM(VOCAB_SIZE).to(DEVICE)
# 固定形状：input_ids.shape = [2, 12]。
input_ids = torch.randint(4, VOCAB_SIZE, (2, TEXT_LENGTH), device=DEVICE)
attention_mask = torch.ones_like(input_ids)
labels = input_ids.clone()
labels[:, :5] = IGNORE_INDEX  # 前 5 个 Token 代表 BOS 与 Prompt；模板、Tokenizer 或监督边界变化时必须重建 Label 并复核有效 Token 分母。
output = model(images, input_ids, attention_mask, labels)
{"logits": tuple(output.logits.shape), "loss": float(output.loss.detach())}


#### 3.3.1．视觉 Token 如何接入语言模型：架构分镜

学习问题是：Projector 与 Resampler 分别改变了视觉表示的哪一条轴，视觉前缀又如何与文本序列形成 Decoder 输入。Projector 执行 `[B, Nv, Dv] → [B, Nv, Dl]`，只对齐特征维度；Resampler 执行 `[B, Nv, Dl] → [B, Nq, Dl]`，再压缩 Token 长度；最终沿序列轴拼接 `[B, Nq, Dl]` 与 `[B, Nt, Dl]`，得到 `[B, Nq + Nt, Dl]`。

下图调用已经定义的 `model.vision_encoder`、`model.resampler.projector`、`model.resampler` 与 `model.token_embedding`，直接提取当前 `images`、`input_ids` 对应的真实中间张量。矩形宽度对应隐藏维度，矩形高度对应 Token 长度；因此 Projector 应只扩展宽度，Resampler 应只缩短高度。


In [ ]:
# 使用同一微型多模态模型的真实中间张量绘制 Connector 架构分镜。
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

connector_model_was_training = model.training
model.eval()
try:
    with torch.inference_mode():
        connector_patch_tokens_device = model.vision_encoder(images)
        connector_projected_memory_device = model.resampler.projector(connector_patch_tokens_device)
        connector_visual_prefix_device = model.resampler(connector_patch_tokens_device)
        connector_text_sequence_device = model.token_embedding(input_ids)
        connector_multimodal_sequence_device = torch.cat(
            [connector_visual_prefix_device, connector_text_sequence_device], dim=1,
        )
finally:
    model.train(connector_model_was_training)

# 可视化 Trace 统一脱离计算图、转 float 并移到 CPU。
connector_patch_tokens = connector_patch_tokens_device.detach().float().cpu()
connector_projected_memory = connector_projected_memory_device.detach().float().cpu()
connector_visual_prefix = connector_visual_prefix_device.detach().float().cpu()
connector_text_sequence = connector_text_sequence_device.detach().float().cpu()
connector_multimodal_sequence = connector_multimodal_sequence_device.detach().float().cpu()
connector_logits = output.logits.detach().float().cpu()

connector_batch_size = images.size(0)
connector_patch_count = (IMAGE_SIZE // PATCH_SIZE) ** 2
connector_text_length = input_ids.size(1)
connector_total_length = NUM_QUERIES + connector_text_length
if tuple(connector_patch_tokens.shape) != (connector_batch_size, connector_patch_count, VISION_DIM):
    raise RuntimeError("Patch Token 未满足 [B, Nv, Dv] 契约")
if connector_projected_memory.shape[:2] != connector_patch_tokens.shape[:2]:
    raise RuntimeError("Projector 错误地改变了 Batch 或视觉 Token 长度")
if connector_projected_memory.size(-1) != LLM_DIM:
    raise RuntimeError("Projector 未把视觉特征维度对齐到 LLM 隐藏维")
if tuple(connector_visual_prefix.shape) != (connector_batch_size, NUM_QUERIES, LLM_DIM):
    raise RuntimeError("Resampler 未把视觉长度压缩为固定 Query 数")
if tuple(connector_text_sequence.shape) != (connector_batch_size, connector_text_length, LLM_DIM):
    raise RuntimeError("文本 Embedding 未满足 [B, Nt, Dl] 契约")
if tuple(connector_multimodal_sequence.shape) != (connector_batch_size, connector_total_length, LLM_DIM):
    raise RuntimeError("视觉前缀与文本序列拼接形状不一致")
if connector_logits.shape[:2] != connector_multimodal_sequence.shape[:2]:
    raise RuntimeError("Decoder logits 未保持多模态输入的 Batch 与序列长度")
connector_prefix_copy_max_abs = float((
    connector_multimodal_sequence[:, :NUM_QUERIES] - connector_visual_prefix
).abs().max())
connector_text_copy_max_abs = float((
    connector_multimodal_sequence[:, NUM_QUERIES:] - connector_text_sequence
).abs().max())
if connector_prefix_copy_max_abs != 0.0 or connector_text_copy_max_abs != 0.0:
    raise RuntimeError("沿 Token 轴拼接时视觉前缀或文本序列发生数值变化")
connector_stage_tensors = {
    "Patch Tokens": connector_patch_tokens,
    "Projected Memory": connector_projected_memory,
    "Visual Prefix": connector_visual_prefix,
    "Text Embedding": connector_text_sequence,
    "Multimodal Sequence": connector_multimodal_sequence,
}
if not all(bool(torch.isfinite(tensor).all()) for tensor in connector_stage_tensors.values()):
    raise RuntimeError("Connector Trace 包含非有限数值")

def my_draw_connector_block(axis, x, y, tensor, title, face_color, max_tokens, max_hidden):
    """按真实 Token 长度和隐藏维度绘制一个形状块，并返回连接锚点。"""
    token_count, hidden_size = tensor.shape[1:]
    width = 1.4 + 1.6 * hidden_size / max_hidden
    height = 0.65 + 1.55 * token_count / max_tokens
    axis.add_patch(Rectangle(
        (x, y), width, height, facecolor=face_color, edgecolor=face_color, alpha=0.34, linewidth=2.0,
    ))
    for row_index in range(1, token_count):
        row_y = y + height * row_index / token_count
        axis.plot([x, x + width], [row_y, row_y], color=face_color, alpha=0.18, linewidth=0.6)
    mean_token_l2 = float(torch.linalg.vector_norm(tensor[0], dim=1).mean())
    axis.text(x + width / 2, y + height + 0.18, title, ha="center", va="bottom", fontweight="bold")
    axis.text(x + width / 2, y + height / 2 + 0.10, f"[B, {token_count}, {hidden_size}]", ha="center", va="center")
    axis.text(x + width / 2, y + height / 2 - 0.22, f"mean ||token||₂={mean_token_l2:.2f}", ha="center", va="center", fontsize=8)
    return {
        "left": (x, y + height / 2), "right": (x + width, y + height / 2),
        "top": (x + width / 2, y + height), "bottom": (x + width / 2, y),
        "width": width, "height": height,
    }

def my_draw_connector_arrow(axis, start, end, title, detail, label_y_offset=0.0):
    """连接两个形状块，并直接标注发生变化的张量轴。"""
    axis.annotate(
        "", xy=end, xytext=start,
        arrowprops={"arrowstyle": "-|>", "linewidth": 1.7, "color": "#374151"},
    )
    middle_x = (start[0] + end[0]) / 2
    middle_y = (start[1] + end[1]) / 2 + label_y_offset
    axis.text(middle_x, middle_y + 0.14, title, ha="center", va="bottom", fontweight="bold", fontsize=9)
    axis.text(middle_x, middle_y - 0.02, detail, ha="center", va="top", fontsize=8)

max_story_tokens = max(tensor.size(1) for tensor in connector_stage_tensors.values())
max_story_hidden = max(tensor.size(2) for tensor in connector_stage_tensors.values())
fig, axis = plt.subplots(figsize=(17, 6.2), constrained_layout=True)
patch_box = my_draw_connector_block(
    axis, 0.4, 3.35, connector_patch_tokens, "Patch Tokens", "#56B4E9", max_story_tokens, max_story_hidden,
)
projected_box = my_draw_connector_block(
    axis, 4.0, 3.35, connector_projected_memory, "Projected Memory", "#0072B2", max_story_tokens, max_story_hidden,
)
prefix_box = my_draw_connector_block(
    axis, 8.0, 4.0, connector_visual_prefix, "Visual Prefix", "#CC79A7", max_story_tokens, max_story_hidden,
)
text_box = my_draw_connector_block(
    axis, 8.0, 0.65, connector_text_sequence, "Text Embedding", "#E69F00", max_story_tokens, max_story_hidden,
)

final_x, final_y = 12.4, 2.55
final_width = 1.4 + 1.6 * LLM_DIM / max_story_hidden
final_height = 0.65 + 1.55 * connector_total_length / max_story_tokens
text_height = final_height * connector_text_length / connector_total_length
prefix_height = final_height - text_height
axis.add_patch(Rectangle(
    (final_x, final_y), final_width, text_height,
    facecolor="#E69F00", edgecolor="none", alpha=0.34,
))
axis.add_patch(Rectangle(
    (final_x, final_y + text_height), final_width, prefix_height,
    facecolor="#CC79A7", edgecolor="none", alpha=0.34,
))
axis.add_patch(Rectangle(
    (final_x, final_y), final_width, final_height,
    facecolor="none", edgecolor="#009E73", linewidth=2.2,
))
axis.plot(
    [final_x, final_x + final_width], [final_y + text_height, final_y + text_height],
    color="#009E73", linewidth=1.5,
)
axis.text(final_x + final_width / 2, final_y + final_height + 0.18, "Multimodal Sequence", ha="center", va="bottom", fontweight="bold")
axis.text(final_x + final_width / 2, final_y + final_height / 2, f"[B, {connector_total_length}, {LLM_DIM}]", ha="center", va="center")
axis.text(final_x + 0.12, final_y + text_height + prefix_height / 2, f"V × {NUM_QUERIES}", ha="left", va="center", fontsize=8)
axis.text(final_x + 0.12, final_y + text_height / 2, f"T × {connector_text_length}", ha="left", va="center", fontsize=8)
final_left_top = (final_x, final_y + text_height + prefix_height / 2)
final_left_bottom = (final_x, final_y + text_height / 2)

my_draw_connector_arrow(
    axis, patch_box["right"], projected_box["left"], "Projector",
    f"D: {VISION_DIM}→{LLM_DIM}；N 保持 {connector_patch_count}", label_y_offset=0.18,
)
my_draw_connector_arrow(
    axis, projected_box["right"], prefix_box["left"], "Resampler",
    f"N: {connector_patch_count}→{NUM_QUERIES}；D 保持 {LLM_DIM}", label_y_offset=0.30,
)
my_draw_connector_arrow(axis, prefix_box["right"], final_left_top, "视觉前缀", "连续向量，不是词表 ID", label_y_offset=0.24)
my_draw_connector_arrow(axis, text_box["right"], final_left_bottom, "沿 Token 轴拼接", f"{NUM_QUERIES}+{connector_text_length}={connector_total_length}", label_y_offset=-0.28)
axis.set_xlim(0.0, 16.0)
axis.set_ylim(0.0, 6.75)
axis.set_aspect("equal")
axis.set_axis_off()
axis.set_title("视觉 Token 接入 Decoder-only LLM：先对齐特征维，再压缩长度，最后拼接序列", pad=18)
plt.show()

connector_mean_token_l2 = {
    name: float(torch.linalg.vector_norm(tensor[0], dim=1).mean())
    for name, tensor in connector_stage_tensors.items()
}
print(
    "Connector 架构分镜摘要：",
    {
        "shape_flow": {name: tuple(tensor.shape) for name, tensor in connector_stage_tensors.items()},
        "projector_token_length_ratio": connector_projected_memory.size(1) / connector_patch_tokens.size(1),
        "resampler_token_length_ratio": connector_visual_prefix.size(1) / connector_projected_memory.size(1),
        "mean_token_l2": connector_mean_token_l2,
        "prefix_concat_max_abs": connector_prefix_copy_max_abs,
        "text_concat_max_abs": connector_text_copy_max_abs,
        "logits_shape": tuple(connector_logits.shape),
    },
)


**应观察到的现象**：Projector 前后的 Token 数保持 16，仅隐藏维由 32 变为 48；Resampler 随后把视觉长度由 16 压缩为 4，但维度继续保持 48。4 个视觉前缀与 12 个文本 Embedding 沿 Token 轴拼接为长度 16 的连续向量序列，拼接前后对应片段的最大绝对误差应严格为 0。

**不可误读的边界**：维度对齐只建立可计算接口，不证明视觉语义已经与语言语义对齐；Resampler 的固定 Query 是学习得到的信息汇聚，不等同于选择 4 个原始 Patch，也不保证压缩无损。视觉前缀是 LLM 隐藏空间中的连续向量，不是 4 个可解码文本 Token。当前随机初始化模型和随机图像只验证架构、形状与拼接契约，不能据此推断视觉理解能力或每个 Query 的可解释语义。


### 3.4．分阶段训练与解冻策略

| 阶段 | 常见数据 | Vision Encoder | Connector | LLM | 主要风险 |
|---|---|---:|---:|---:|---|
| 模态对齐 | 图文对、Caption | 冻结 | 训练 | 冻结 | Connector 只学到表面词汇映射 |
| 多模态预训练 | 大规模交错图文 | 冻结或后段解冻 | 训练 | 部分/全部训练 | 数据质量、版权、OCR 污染 |
| 多模态 SFT | 图文指令与答案 | 通常冻结 | 训练 | PEFT 或部分训练 | Label 泄漏、模板漂移 |
| 偏好/安全对齐 | 成对偏好、拒答和安全集 | 冻结 | 视任务而定 | PEFT/策略训练 | Reward 投机与跨模态越狱 |

视觉预处理器、图像尺寸、Patch 策略、特殊 Image Token、Chat Template 和 LLM Tokenizer 都属于同一个不可拆分的模型制品。


## 4．证据验证

原理实现的验收证据包括：

1. `vision_tokens` 形状为 `[B, 16, 32]`，`visual_prefix` 为 `[B, 4, 48]`，证明 Projector 对齐维度且 Resampler 压缩长度。
2. 拼接后的 Logits 形状为 `[B, NUM_QUERIES + TEXT_LENGTH, VOCAB_SIZE]`。
3. 视觉位置和 Prompt 位置的 Label 为 `IGNORE_INDEX`，但 Attention Mask 仍为有效，证明“可读取但不监督”。
4. 因果 Mask 阻止当前位置读取未来答案；更换图像时，视觉前缀和后续 Logits 应发生可观察变化。
5. 标准库迁移后，用同一版本化图文夹具检查模板、视觉 Token 数、输出 Schema 和确定性生成回归，并同时验证运行结果的语义一致性。


### 4.1．视觉 Token 流与 Label Mask 证据图

**学习问题。** 当前图像如何从 Patch Grid 变成视觉 Token，经 Projector 对齐 LLM 维度、由 Resampler 压缩长度，再与文本拼接？视觉与 Prompt 位置为什么可以被 Decoder 读取，却不参与语言建模损失？

本图直接复用前文的 `model`、`images`、`input_ids`、`attention_mask`、`labels` 与 `output`，并调用模型内部的 `vision_encoder`、`resampler.projector`、`resampler` 和 `token_embedding` 暴露同一次数据契约中的真实中间张量。上方两图分别展示每阶段的序列长度与隐藏维度；下方二值图对照多模态 Attention Mask 和 Label 监督位置。

运行前应明确以下形状与数值不变量：

- Patch Grid 为 `[B, Dv, H/P, W/P] = [2, 32, 4, 4]`，展平后 `vision_tokens` 为 `[B, Nv, Dv] = [2, 16, 32]`，且 `Nv=(H/P)(W/P)=16`。
- Projector 只把最后一维从 `Dv=32` 变为 `Dl=48`，不改变 `[B, Nv]`；Resampler 再把长度从 `Nv=16` 压缩为 `Nq=4`，输出 `[2, 4, 48]`。
- 文本 Embedding 为 `[B, Nt, Dl] = [2, 12, 48]`，拼接序列和 logits 的长度都必须为 `Nq+Nt=16`；logits 最后一维为 `VOCAB_SIZE=128`。
- 多模态 Attention Mask 与 Label Mask 都具有 `[B, Nq+Nt]` 形状。当前夹具中 4 个视觉位置和前 5 个 Prompt 位置的 Attention 值为 1、Label 为 `IGNORE_INDEX`；余下 7 个 Answer 位置参与 next-token loss。


In [ ]:
# 从当前模型组件提取真实中间张量，并复现 forward 中的拼接与 Mask 契约。
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

with torch.no_grad():
    trace_patch_grid = model.vision_encoder.patch_embedding(images)
    trace_vision_tokens = model.vision_encoder(images)
    trace_projected_memory = model.resampler.projector(trace_vision_tokens)
    trace_visual_prefix = model.resampler(trace_vision_tokens)
    trace_text_tokens = model.token_embedding(input_ids)
    trace_multimodal_sequence = torch.cat(
        [trace_visual_prefix, trace_text_tokens], dim=1
    )

trace_visual_attention_mask = torch.ones(
    attention_mask.size(0), NUM_QUERIES,
    dtype=attention_mask.dtype, device=attention_mask.device,
)
trace_multimodal_attention_mask = torch.cat(
    [trace_visual_attention_mask, attention_mask], dim=1
)
trace_visual_labels = torch.full(
    (labels.size(0), NUM_QUERIES), IGNORE_INDEX,
    dtype=labels.dtype, device=labels.device,
)
trace_multimodal_labels = torch.cat([trace_visual_labels, labels], dim=1)

batch_size = images.size(0)
patch_grid_token_count = trace_patch_grid.size(2) * trace_patch_grid.size(3)
total_sequence_length = NUM_QUERIES + input_ids.size(1)
if tuple(trace_vision_tokens.shape) != (batch_size, patch_grid_token_count, VISION_DIM):
    raise RuntimeError("Patch Grid 展平后的视觉 Token 形状不一致")
if trace_projected_memory.shape[:2] != trace_vision_tokens.shape[:2] or trace_projected_memory.size(-1) != LLM_DIM:
    raise RuntimeError("Projector 未保持 [B, Nv] 或未对齐 LLM 隐藏维")
if tuple(trace_visual_prefix.shape) != (batch_size, NUM_QUERIES, LLM_DIM):
    raise RuntimeError("Resampler 输出未满足 [B, Nq, Dl] 契约")
if tuple(trace_multimodal_sequence.shape) != (batch_size, total_sequence_length, LLM_DIM):
    raise RuntimeError("视觉前缀与文本 Embedding 拼接形状不一致")
if tuple(output.logits.shape) != (batch_size, total_sequence_length, VOCAB_SIZE):
    raise RuntimeError("模型 logits 未保持多模态序列长度或词表维")
if trace_multimodal_attention_mask.shape != trace_multimodal_labels.shape:
    raise RuntimeError("Attention Mask 与 Label Mask 形状不一致")

prompt_token_count = int(labels[0].eq(IGNORE_INDEX).sum().item())
answer_token_count = input_ids.size(1) - prompt_token_count
ignored_prefix_length = NUM_QUERIES + prompt_token_count
if not bool(trace_multimodal_attention_mask[:, :ignored_prefix_length].bool().all()):
    raise RuntimeError("视觉或 Prompt 位置没有保持可读取状态")
if not bool(trace_multimodal_labels[:, :ignored_prefix_length].eq(IGNORE_INDEX).all()):
    raise RuntimeError("视觉或 Prompt 位置错误地进入语言损失")
if not bool(trace_multimodal_labels[:, ignored_prefix_length:].ne(IGNORE_INDEX).all()):
    raise RuntimeError("Answer 位置未完整进入语言损失")

stage_names = ["Patch\nTokens", "Projected\nMemory", "Visual\nPrefix", "Text\nEmbedding", "Multimodal\nSequence"]
stage_tensors = [
    trace_vision_tokens, trace_projected_memory, trace_visual_prefix,
    trace_text_tokens, trace_multimodal_sequence,
]
stage_token_counts = [tensor.size(1) for tensor in stage_tensors]
stage_hidden_dims = [tensor.size(2) for tensor in stage_tensors]
stage_colors = ["#0ea5e9", "#2563eb", "#7c3aed", "#0f766e", "#ea580c"]

fig = plt.figure(figsize=(15, 8.5), constrained_layout=True)
grid = fig.add_gridspec(2, 2, height_ratios=(1.0, 1.25))
length_ax = fig.add_subplot(grid[0, 0])
dimension_ax = fig.add_subplot(grid[0, 1])
mask_ax = fig.add_subplot(grid[1, :])

length_bars = length_ax.bar(stage_names, stage_token_counts, color=stage_colors)
length_ax.bar_label(length_bars, labels=[str(value) for value in stage_token_counts])
length_ax.set_ylabel("Token 数")
length_ax.set_title(
    f"序列长度流：Patch Grid {trace_patch_grid.size(2)}×{trace_patch_grid.size(3)}"
)
length_ax.grid(axis="y", alpha=0.25)

dimension_bars = dimension_ax.bar(stage_names, stage_hidden_dims, color=stage_colors)
dimension_ax.bar_label(dimension_bars, labels=[str(value) for value in stage_hidden_dims])
dimension_ax.set_ylabel("隐藏维度")
dimension_ax.set_title("Projector 对齐 Dv→Dl；Resampler 保持 Dl")
dimension_ax.grid(axis="y", alpha=0.25)

mask_rows = torch.stack(
    [
        trace_multimodal_attention_mask[0].bool(),
        trace_multimodal_labels[0].ne(IGNORE_INDEX),
    ]
).to(dtype=torch.int).detach().cpu()
position_labels = (
    [f"V{index + 1}" for index in range(NUM_QUERIES)]
    + [f"P{index + 1}" for index in range(prompt_token_count)]
    + [f"A{index + 1}" for index in range(answer_token_count)]
)
mask_image = mask_ax.imshow(
    mask_rows, cmap=ListedColormap(["#e2e8f0", "#2563eb"]),
    vmin=0, vmax=1, aspect="auto", interpolation="nearest",
)
for row_index in range(mask_rows.size(0)):
    for column_index in range(mask_rows.size(1)):
        value = int(mask_rows[row_index, column_index])
        mask_ax.text(
            column_index, row_index, str(value), ha="center", va="center",
            color="white" if value else "#334155", fontsize=9,
        )
mask_ax.set_xticks(range(total_sequence_length), position_labels)
mask_ax.set_yticks([0, 1], ["Attention：可读取", "Label：参与 loss"] )
mask_ax.set_xlabel("多模态序列位置（V=视觉，P=Prompt，A=Answer）")
mask_ax.set_title("同一序列中的 Attention Mask 与 Label 监督边界")
mask_colorbar = fig.colorbar(mask_image, ax=mask_ax, ticks=[0, 1], shrink=0.78)
mask_colorbar.ax.set_yticklabels(["0：无效/忽略", "1：有效/监督"])
fig.suptitle("真实视觉张量到多模态序列与 Label Mask 的 Token 流")
plt.show()

print(
    "多模态 Token 流契约：",
    {
        "patch_grid": tuple(trace_patch_grid.shape),
        "vision_tokens": tuple(trace_vision_tokens.shape),
        "projected_memory": tuple(trace_projected_memory.shape),
        "visual_prefix": tuple(trace_visual_prefix.shape),
        "multimodal_sequence": tuple(trace_multimodal_sequence.shape),
        "attention_mask": tuple(trace_multimodal_attention_mask.shape),
        "label_mask": tuple(trace_multimodal_labels.shape),
        "supervised_tokens_per_sample": answer_token_count,
    },
)


**应观察到的结论。** Projector 柱只改变隐藏维度，序列长度仍为 16；Resampler 把视觉长度从 16 压缩为 4，同时保持 LLM 隐藏维 48。视觉前缀与 12 个文本 Embedding 拼接后形成长度 16 的 Decoder 输入。Mask 图中 V1～V4 与 P1～P5 在 Attention 行为 1、在 Label 行为 0，说明这些上下文可被后续位置读取，却不直接承担语言损失；A1～A7 同时可读取并参与监督。

**不可误读的边界。** 这些张量来自随机图像和随机初始化的微型模型，只证明形状、连接和监督协议，不证明模型已获得视觉语义、定位或 OCR 能力。Patch Token、Projected Memory 和 Visual Prefix 都是连续向量，不是可直接解码的离散 Token ID；Resampler 缩短长度也不保证无损保留信息。Label Mask 与 Attention Mask 解决不同问题，前者为 0 不能推断该位置对预测没有影响。当前夹具没有 padding、多图、动态分辨率或视频帧，生产视觉 Token 数仍由绑定版本的 Processor、切块/缩放策略和模型 Config 决定。


## 5．迁移到生产库

本节采用小型 VLM 的当前仓库文件 验证生产库接口。首次运行会下载模型；本地原理实现不依赖该外部资产。`Processor` 同时负责图像预处理、Chat Template 和 Tokenizer，不能只保存模型权重。


In [ ]:
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    AutoModelForImageTextToText, AutoProcessor, LogitsProcessor, LogitsProcessorList,
)

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
# 32 个新 Token 仅限制接口验证时延；生产上限应按任务长度和 SLO 校准。
MAX_NEW_TOKENS = 32

processor = AutoProcessor.from_pretrained(MODEL_ID)
library_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype="auto",
).to(DEVICE)

# 256×256 纯色图像是可复现的 Processor 接口夹具；生产尺寸由 max pixels 与质量—成本曲线确定。
image = Image.new("RGB", (256, 256), color=(40, 90, 160))
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "请简洁描述这张图片的主要颜色。"},
        ],
    }
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {name: value.to(DEVICE) for name, value in inputs.items()}


class MyGenerationProgress(LogitsProcessor):
    """按多模态解码步显示进度，并保持 logits 不变。"""
    def __init__(self, total):
        self.progress = tqdm(
            total=total, desc="生成多模态回答", unit="token-step", dynamic_ncols=True
        )

    def __call__(self, input_ids, scores):
        self.progress.update(1)
        return scores

    def close(self):
        self.progress.close()
# do_sample=False 使用 Greedy 解码以支持确定性回归；启用采样时需同时固定 temperature、top-p/top-k、stop 与 seed。
generation_progress = MyGenerationProgress(MAX_NEW_TOKENS)
try:
    generated_ids = library_model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        logits_processor=LogitsProcessorList([generation_progress]),
    )
finally:
    generation_progress.close()
new_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
processor.batch_decode(new_ids, skip_special_tokens=True)


## 6．生产边界

1. 固定权重、Vision/Audio Encoder、Connector、LLM、Processor、Tokenizer、Chat Template、精度与后端版本。
2. **理解**分别评测 OCR、图表、空间关系、时序定位、多图/多帧一致性、引用区域和纯文本回退；不把语言流畅度当视觉事实正确率。
3. **图像生成**评测提示遵循、文字/布局、编辑保持、人物/品牌安全、延迟与峰值显存；**视频生成**另评运动、身份、镜头、时序、声画同步与物理一致性。
4. 记录分辨率、帧率、媒体时长、视觉/音频 Token、Prefill/生成时延、峰值显存和单位请求成本，并在网关强制预算。
5. 检查 EXIF、隐写、恶意文档/音频、跨模态 Prompt Injection、声音/肖像冒充、版权和训练数据来源；生成内容建立水印/C2PA 或等价来源追踪。
6. 对无法识别、低置信度、超出政策范围或将触发物理/业务动作的输出建立拒答、人工升级或受控工具执行路径。世界模型的预测不能直接成为安全关键控制信号。
7. 先记录权重/代码可取得或 API-only 等**可见性状态**，再用许可证矩阵逐项核验：修改、再分发、商业部署、MaaS、地域、收入/用户门槛、衍生训练、用输出训练、归因和 AUP。不可把这些条件压成线性“开放等级”。

### 6.1．官方资料

- 理解/Omni：[Qwen3-VL](https://huggingface.co/collections/Qwen/qwen3-vl)、[Qwen3-Omni](https://github.com/QwenLM/Qwen3-Omni)、[Kimi-VL](https://github.com/MoonshotAI/Kimi-VL)、[Kimi K3](https://github.com/MoonshotAI/Kimi-K3)、[InternVL](https://github.com/OpenGVLab/InternVL)
- 美国模型：[Llama 4 Model Card](https://github.com/meta-llama/llama-models/blob/main/models/llama4/MODEL_CARD.md)、[Gemma 4 Model Card](https://ai.google.dev/gemma/docs/core/model_card_4)、[Phi-4 Multimodal](https://huggingface.co/microsoft/Phi-4-multimodal-instruct)
- 理解与生成统一：[DeepSeek Janus](https://github.com/deepseek-ai/Janus)
- 生成路线入口：[Qwen-Image](https://github.com/QwenLM/Qwen-Image)、[Wan2.2](https://github.com/Wan-Video/Wan2.2)、[HunyuanVideo-1.5](https://github.com/Tencent-Hunyuan/HunyuanVideo-1.5)、[Mochi](https://github.com/genmoai/mochi)、[NVIDIA Cosmos](https://github.com/NVIDIA/cosmos)

ViT 视觉 Token 接入 LLM 与 Diffusion/Flow 生成属于不同任务链路，两者的目标、许可证和部署指标应分别验收。
